## Install Dependencies

In [ ]:
%pip install -U diffusers transformers accelerate mlflow torch torchvision

In [ ]:
dbutils.library.restartPython()

## Download Model from HuggingFace

In [ ]:
from huggingface_hub import snapshot_download
import os

model_path = "Qwen/Qwen-Image-Edit-2509"
model_cache_path = f"/local_disk0/models_cache/{model_path}"
download_models = True

if download_models:
    os.environ["HF_HOME"] = "/local_disk0/hf"
    os.environ["HF_TOKEN"] = dbutils.secrets.get(
        scope="shj_db_scope", key="hf_secret"
    )
    
    snapshot_location = snapshot_download(
        repo_id=model_path,
        local_dir=model_cache_path,
        ignore_patterns="*.pth"
    )
else:
    snapshot_location = model_cache_path

snapshot_location

## Create Inference Config

In [ ]:
import yaml

inference_config = {
    "model_type": "qwen_image_edit",
    "use_quantization": False
}

with open('inference_config.yml', 'w') as f:
    yaml.dump(inference_config, f)

## Prepare Input Examples and Signature

In [ ]:
from mlflow.models.signature import infer_signature
from PIL import Image
import io
import base64
import pandas as pd

def pillow_image_to_base64_string(img):
    buffered = io.BytesIO()
    img.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

# Load example images
image1 = Image.open("sample_images/01_ice-castle-image.png").convert("RGB")
image2 = Image.open("sample_images/02_brown-bear-image.png").convert("RGB")

# Convert to base64
image1_base64 = pillow_image_to_base64_string(image1)
image2_base64 = pillow_image_to_base64_string(image2)

# Create input example
input_example = pd.DataFrame().from_records([{
    "image1": image1_base64,
    "image2": image2_base64,
    "prompt": "The magician emperor bear is standing in front of castle with a diamond topped septar in his hand. Keep a snowing background and a blue sky."
}])

# Define parameters
params = {
    "num_inference_steps": 40,
    "true_cfg_scale": 4.0,
    "guidance_scale": 1.0,
    "negative_prompt": " ",
    "num_images_per_prompt": 1,
    "seed": 0
}

# Create output example
output_example = pd.DataFrame().from_records([{
    "output_image": "this is an example base64 output image"
}])

signature = infer_signature(input_example, output_example, params)
print(signature)

## Log Model to MLflow

In [ ]:
import mlflow

ds_model_path = os.path.join(os.getcwd(), "imagen_model.py")
config_path = os.path.join(os.getcwd(), "inference_config.yml")

with mlflow.start_run():
    model_info = mlflow.pyfunc.log_model(
        "model",
        python_model=ds_model_path,
        model_config=config_path,
        artifacts={"model_path": model_cache_path},
        input_example=input_example,
        signature=signature,
        pip_requirements=[
            "diffusers",
            "transformers",
            "torch",
            "torchvision",
            "accelerate",
            "mlflow"
        ]
    )

model_info.model_uri

## Test the Logged Model

In [ ]:
dbutils.library.restartPython()

In [ ]:
import mlflow
import pandas as pd
from PIL import Image
import io
import base64

def pillow_image_to_base64_string(img):
    buffered = io.BytesIO()
    img.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

def base64_string_to_pillow_image(base64_str):
    return Image.open(io.BytesIO(base64.decodebytes(bytes(base64_str, "utf-8"))))

# Load example images
image1 = Image.open("sample_images/01_ice-castle-image.png").convert("RGB")
image2 = Image.open("sample_images/02_brown-bear-image.png").convert("RGB")

# Convert to base64
image1_base64 = pillow_image_to_base64_string(image1)
image2_base64 = pillow_image_to_base64_string(image2)

# Create input
input_example = pd.DataFrame().from_records([{
    "image1": image1_base64,
    "image2": image2_base64,
    "prompt": "The magician emperor bear is standing in front of castle with a diamond topped septar in his hand. Keep a snowing background and a blue sky. Do not include any bookmarks"
}])

In [ ]:
# Load the model (replace with your actual model URI from above)
model_uri = 'runs:/<run_id>/model'
loaded_model = mlflow.pyfunc.load_model(model_uri)

In [ ]:
# Predict
output = loaded_model.predict(
    input_example,
    params={
        "num_inference_steps": 40,
        "true_cfg_scale": 4.0,
        "guidance_scale": 1.0,
        "negative_prompt": " ",
        "num_images_per_prompt": 1,
        "seed": 0
    }
)

# Display the output image
output_image_base64 = output.iloc[0]["output_image"]
output_image = base64_string_to_pillow_image(output_image_base64)
display(output_image)

In [ ]:
# Save the output image
output_image.save("sample_images/output_image_mlflow_test.png")
print("Image saved at sample_images/output_image_mlflow_test.png")

## Register Model to Unity Catalog

In [ ]:
mlflow.set_registry_uri("databricks-uc")
registered_model = mlflow.register_model(
    model_info.model_uri,
    "uc_sriharsha_jana.test_db.qwen_image_edit_model"
)